In [2]:
import numpy as np
import pandas as pd
import statsmodels.tsa.stattools as ts
from pathlib import Path                    # Obsługa projektu
from scipy import stats
import statsmodels.api as sm

In [3]:
# ── Konfiguracja 2: Wczytanie danych z 01_etl ──────────────────────

# Konfiguracja ścieżek (zgodna z pierwszym notebookiem)
PROJECT_ROOT = Path.cwd().parent  # ponieważ notebook jest w notebooks/
DATA_DIR = PROJECT_ROOT / "processed"  # uwaga: bez "data" w środku

# Odczyt danych
df = pd.read_parquet(DATA_DIR / "df.parquet")
df_rok = pd.read_parquet(DATA_DIR / "df_rok.parquet")

In [4]:
# Test stacjonarności

adf = ts.adfuller(df['vat_pkb_pct'],
                  maxlag=0,
                  regression='ct', # trend i wyraz wolny
                  autolag=None,
                  store=False,
                  regresults=False)

print(f'ADF statystyka:     {adf[0]:.4f}')
print(f'p-value:            {adf[1]:.4f}')
print(f"Użyte opóźnienia:   {adf[2]}")
print(f"Liczba obserwacji:  {adf[3]}")
print('Krytyczne wartości:')
for key, value in adf[4].items():
    print(f'  {key}: {value:.4f}')


alpha = 0.05

print("\nWynik testu ADF:")
if adf[1] < alpha:
    print("Odrzucamy hipotezę zerową (H0) na rzecz hipotezy alternatywnej (H1)")
    print("Szereg jest stacjonarny.")
else:
    print("Brak podstaw do odrzucenia hipotezy zerowej (H0).")
    print("Szereg jest niestacjonarny.")

ADF statystyka:     -7.1716
p-value:            0.0000
Użyte opóźnienia:   0
Liczba obserwacji:  107
Krytyczne wartości:
  1%: -4.0460
  5%: -3.4523
  10%: -3.1516

Wynik testu ADF:
Odrzucamy hipotezę zerową (H0) na rzecz hipotezy alternatywnej (H1)
Szereg jest stacjonarny.


In [5]:
# Test na normalność - Shapiro - Wilk

# Podziel dane na grupy przed i po reformie
df_before = df.loc[df.index <= '2016Q2', 'vat_pkb_pct']  # to już jest Series
df_after  = df.loc[df.index >= '2016Q3', 'vat_pkb_pct']  # to już jest Series

# ============================================
# TEST SHAPIRO-WILKA dla grupy PRZED reformą
# ============================================
shapiro_stat_before, shapiro_p_before = stats.shapiro(df_before)

print('='*50)
print('Test Shapiro-Wilk - grupa PRZED reformą (do 2016-Q2)')
print('='*50)
print(f'Statystyka: {shapiro_stat_before:.4f}')
print(f'p-value:    {shapiro_p_before:.4f}')

alpha = 0.05
print(f'\nWynik (alpha={alpha}):')
if shapiro_p_before < alpha:
    print("  ODRZUCAMY H0 - rozkład NIE jest normalny")
else:
    print("  NIE ODRZUCAMY H0 - rozkład jest normalny")

# ============================================
# TEST SHAPIRO-WILKA dla grupy PO reformie
# ============================================
shapiro_stat_after, shapiro_p_after = stats.shapiro(df_after)

print('\n' + '='*50)
print('Test Shapiro-Wilk - grupa PO reformie (od 2016-Q3)')
print('='*50)
print(f'Statystyka: {shapiro_stat_after:.4f}')
print(f'p-value:    {shapiro_p_after:.4f}')

if shapiro_p_after < alpha:
    print("  ODRZUCAMY H0 - rozkład NIE jest normalny")
else:
    print("  NIE ODRZUCAMY H0 - rozkład jest normalny")

# ============================================
# PODSUMOWANIE - co dalej?
# ============================================
print('\n' + '='*50)
print('PODSUMOWANIE DLA KROKU 2')
print('='*50)

if shapiro_p_before >= alpha and shapiro_p_after >= alpha:
    print("OBIE grupy mają rozkład normalny → używamy testu t (parametryczny)")
else:
    print("Przynajmniej JEDNA grupa nie ma rozkładu normalnego → używamy Mann-Whitney U (nieparametryczny)")

Test Shapiro-Wilk - grupa PRZED reformą (do 2016-Q2)
Statystyka: 0.9921
p-value:    0.9421

Wynik (alpha=0.05):
  NIE ODRZUCAMY H0 - rozkład jest normalny

Test Shapiro-Wilk - grupa PO reformie (od 2016-Q3)
Statystyka: 0.9813
p-value:    0.7638
  NIE ODRZUCAMY H0 - rozkład jest normalny

PODSUMOWANIE DLA KROKU 2
OBIE grupy mają rozkład normalny → używamy testu t (parametryczny)


In [6]:
# Test Levene'a
levene_stat, levene_p = stats.levene(df_before, df_after)

print('='*50)
print('TEST Levene\'a (jednorodność wariancji)')
print('='*50)
print(f'Statystyka: {levene_stat:.4f}')
print(f'p-value:    {levene_p:.4f}')

if levene_p < alpha:
    print("  Wariancje NIE są równe")
else:
    print("  Wariancje są równe")

TEST Levene'a (jednorodność wariancji)
Statystyka: 1.8492
p-value:    0.1768
  Wariancje są równe


In [7]:
# Test t-studenta

t_stat, t_p = stats.ttest_ind(df_before, df_after, equal_var=True)

print('='*50)
print('TEST t (Student, wariancje równe)')
print('='*50)
print(f'Statystyka t: {t_stat:.4f}')
print(f'p-value:      {t_p:.4f}')

if t_p < 0.05:
    print("  ODRZUCAMY H0 - ISTNIEJE istotna różnica między grupami")
else:
    print("  NIE ODRZUCAMY H0 - NIE MA istotnej różnicy między grupami")

TEST t (Student, wariancje równe)
Statystyka t: -4.2592
p-value:      0.0000
  ODRZUCAMY H0 - ISTNIEJE istotna różnica między grupami


In [8]:
# Test COHEN's

# ============================================
# COHEN'S d (wielkość efektu)
# ============================================

n1 = len(df_before)
n2 = len(df_after)
mean1 = np.mean(df_before)
mean2 = np.mean(df_after)
std1 = np.std(df_before, ddof=1)
std2 = np.std(df_after, ddof=1)

# Odchylenie standardowe pooled (dla równych wariancji)
pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))

cohens_d = (mean2 - mean1) / pooled_std

print('='*50)
print('COHEN\'S d - wielkość efektu')
print('='*50)
print(f'Cohen\'s d: {cohens_d:.4f}')
print(f'Interpretacja:', end=' ')

d_abs = abs(cohens_d)
if d_abs < 0.2:
    print("efekt BARDZO MAŁY")
elif d_abs < 0.5:
    print("efekt MAŁY")
elif d_abs < 0.8:
    print("efekt ŚREDNI")
else:
    print("efekt DUŻY")

# Dodatkowo: przedział ufności dla Cohen's d (opcjonalnie)
se_d = np.sqrt((n1 + n2) / (n1 * n2) + cohens_d**2 / (2 * (n1 + n2)))
ci_lower = cohens_d - 1.96 * se_d
ci_upper = cohens_d + 1.96 * se_d
print(f'95% przedział ufności: [{ci_lower:.4f}, {ci_upper:.4f}]')

COHEN'S d - wielkość efektu
Cohen's d: 0.8582
Interpretacja: efekt DUŻY
95% przedział ufności: [0.4470, 1.2694]


In [9]:
# DODATKOWE ZMIENNE

df['vat_pkb_lag1'] = df['vat_pkb_pct'].shift(1)
df['vat_pkb_lag4'] = df['vat_pkb_pct'].shift(4)

df['Q1'] = df.index.str.endswith('Q1').astype(int)
df['Q2'] = df.index.str.endswith('Q2').astype(int)
df['Q3'] = df.index.str.endswith('Q3').astype(int)

df_ar = df[['vat_pkb_pct', 'vat_pkb_lag1', 'vat_pkb_lag4',
             'trend', 'Q1', 'Q2', 'Q3',
             'jpk_2016', 'split_2018',
             'wlist_2019', 'covid_2020', 'cit_mld', 'akcyza_mld']].dropna()

y_ar = df_ar['vat_pkb_pct']

In [10]:
# ── MODEL FINALNY — uproszczony ───────────────────────────────────────────────
X_fin = sm.add_constant(df_ar[['vat_pkb_lag1', 'vat_pkb_lag4',
                                 'Q1', 'Q2', 'Q3',
                                 'jpk_2016', 'covid_2020']])

model_fin     = sm.OLS(y_ar, X_fin).fit()
model_fin_hac = model_fin.get_robustcov_results(cov_type='HAC', maxlags=4)

nazwy_fin = ['stała', 'AR(1)', 'AR(4)', 'Q1', 'Q2', 'Q3',
             'JPK_2016', 'COVID_2020']

print('=' * 72)
print('  MODEL FINALNY — vat_pkb_pct')
print('  Model: vat_pkb_pct = α + AR(1) + AR(4) + Q1 + Q2 + Q3')
print('                     + JPK_2016 + COVID_2020 + ε')
print('  HAC Newey-West SE, maxlags=4')
print('=' * 72)
print(f'  n          = {int(model_fin.nobs)}')
print(f'  R²         = {model_fin.rsquared:.4f}')
print(f'  Adj. R²    = {model_fin.rsquared_adj:.4f}')
print(f'  AIC        = {model_fin.aic:.2f}')
print(f'  BIC        = {model_fin.bic:.2f}')
print()

print(f'  {"Zmienna":<12} {"Coef":>8} {"HAC SE":>8} '
      f'{"t":>7} {"p":>8} {"Istotność":>10}')
print('  ' + '-' * 60)

for nazwa, coef, se, t, p in zip(
    nazwy_fin,
    model_fin_hac.params,
    model_fin_hac.bse,
    model_fin_hac.tvalues,
    model_fin_hac.pvalues,
):
    gwiazdki = ('***' if p<0.01 else '**' if p<0.05 else
                '*'   if p<0.10 else '')
    print(f'  {nazwa:<12} {coef:>8.4f} {se:>8.4f} '
          f'{t:>7.3f} {p:>8.4f} {gwiazdki:>10}')

# ── DIAGNOSTYKA ───────────────────────────────────────────────────────────────
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox

residua_fin = model_fin.resid
dw_fin      = durbin_watson(residua_fin)
_, p_jb_fin, _, _ = sm.stats.stattools.jarque_bera(residua_fin)
_, p_bp_fin, _, _ = het_breuschpagan(residua_fin, X_fin)
lb_fin            = acorr_ljungbox(residua_fin, lags=[4, 8], return_df=True)

print(f'\n=== DIAGNOSTYKA MODELU FINALNEGO ===')
print(f'  Durbin-Watson      : {dw_fin:.3f}  '
      f'({"✅" if 1.5 < dw_fin < 2.5 else "⚠️"})')
print(f'  Jarque-Bera p      : {p_jb_fin:.4f}  '
      f'({"✅" if p_jb_fin > 0.05 else "⚠️"})')
print(f'  Breusch-Pagan p    : {p_bp_fin:.4f}  '
      f'({"✅" if p_bp_fin > 0.05 else "⚠️"})')
print(f'  Ljung-Box lag=4 p  : {lb_fin["lb_pvalue"].iloc[0]:.4f}  '
      f'({"✅" if lb_fin["lb_pvalue"].iloc[0] > 0.05 else "⚠️"})')
print(f'  Ljung-Box lag=8 p  : {lb_fin["lb_pvalue"].iloc[1]:.4f}  '
      f'({"✅" if lb_fin["lb_pvalue"].iloc[1] > 0.05 else "⚠️"})')

  MODEL FINALNY — vat_pkb_pct
  Model: vat_pkb_pct = α + AR(1) + AR(4) + Q1 + Q2 + Q3
                     + JPK_2016 + COVID_2020 + ε
  HAC Newey-West SE, maxlags=4
  n          = 104
  R²         = 0.3656
  Adj. R²    = 0.3193
  AIC        = 165.66
  BIC        = 186.81

  Zmienna          Coef   HAC SE       t        p  Istotność
  ------------------------------------------------------------
  stała          3.8481   1.0287   3.741   0.0003        ***
  AR(1)          0.2706   0.1312   2.062   0.0419         **
  AR(4)          0.2071   0.0945   2.191   0.0309         **
  Q1             0.0083   0.1739   0.048   0.9621           
  Q2            -0.1962   0.1221  -1.607   0.1114           
  Q3            -0.0103   0.1088  -0.094   0.9251           
  JPK_2016       0.3345   0.1552   2.155   0.0337         **
  COVID_2020    -1.5162   0.1611  -9.414   0.0000        ***

=== DIAGNOSTYKA MODELU FINALNEGO ===
  Durbin-Watson      : 1.955  (✅)
  Jarque-Bera p      : 0.0702  (✅)
  Breus